In [ ]:
import os
import pathlib
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model # type: ignore
from tensorflow.keras.metrics import MeanIoU # type: ignore
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support
from dotenv import load_dotenv

# Caminho para o modelo salvo
MODEL_PATH = "deeplabv3plusF_2025-03-27_13-34-11.h5"

# Configurações
NUM_CLASSES = 22
BATCH_SIZE = 8
IMG_SIZE = (512, 512)

def carregar_modelo():
    """Carrega o modelo salvo com métricas personalizadas."""
    # Se você usou métricas personalizadas, precisará defini-las aqui
    custom_objects = {
        # Adicione métricas personalizadas se necessário
    }
    
    try:
        modelo = load_model(MODEL_PATH, custom_objects=custom_objects)
        print(f"Modelo carregado com sucesso: {MODEL_PATH}")
        return modelo
    except Exception as e:
        print(f"Erro ao carregar o modelo: {str(e)}")
        return None

# Conjunto de Dados
def preparar_dataset_teste(img_size=IMG_SIZE, batch_size=BATCH_SIZE):
    """
    Carrega o conjunto de dados de teste
    """
    # Carregar variáveis de ambiente
    load_dotenv()
    
    img_dir_str = os.getenv("BASE_IMG_FOLDER")
    if img_dir_str is None:
        raise ValueError("A variável de ambiente BASE_IMG_FOLDER não está definida no arquivo .env")
    img_dir = pathlib.Path(img_dir_str)
    
    mask_dir_str = os.getenv("NUMPY_FOLDER")
    if mask_dir_str is None:
        raise ValueError("A variável de ambiente NUMPY_FOLDER não está definida no arquivo .env")
    mask_dir = pathlib.Path(mask_dir_str)
    
    # Funções para extrair número do arquivo
    def get_file_number(filepath):
        return int(os.path.basename(filepath).replace('.png', ''))
    
    def get_file_number_mask(filepath):
        return int(os.path.basename(filepath).replace('.npy', ''))
    
    # Listar arquivos
    mask_files = [str(path) for path in mask_dir.glob('*.npy') if os.path.exists(path)]
    image_files = [str(path) for path in img_dir.glob('*.png') if os.path.exists(path)]
    
    # Ordenar
    image_files = sorted(image_files, key=get_file_number)
    mask_files = sorted(mask_files, key=get_file_number_mask)
    
    def load_and_preprocess_image_mask(image_path, mask_path):
        try:
            # Carregar imagem
            img = tf.io.read_file(image_path)
            img = tf.image.decode_png(img, channels=3)
            img = tf.image.resize(img, [512, 512])
            img = tf.cast(img, tf.float32) / 255.0
            
            # Carregar máscara
            mask = np.load(mask_path.numpy().decode())
            mask = tf.convert_to_tensor(mask, dtype=tf.float32)
            mask = mask / 255.0
            mask = tf.where(mask >= 0.3, 1.0, 0.0)
            
            return img, mask
        except Exception as e:
            print(f"Erro ao processar {image_path}, {mask_path}: {str(e)}")
            raise
    
    def process_path(image_path, mask_path):
        img, mask = tf.py_function(load_and_preprocess_image_mask, [image_path, mask_path], [tf.float32, tf.float32])
        # img.set_shape((512, 512, 3))
        # mask.set_shape((512, 512, 22))
        img.set_shape(img_size + (3,))
        mask.set_shape(img_size + (NUM_CLASSES,))
        return img, mask
    
    # Criar dataset
    dataset = tf.data.Dataset.from_tensor_slices((image_files, mask_files))
    dataset = dataset.map(process_path, num_parallel_calls=tf.data.AUTOTUNE)
    
    # Dividir dataset (mesma lógica do código original)
    dataset_size = tf.data.experimental.cardinality(dataset).numpy()
    train_size = int(0.8 * dataset_size)
    val_size = int(0.1 * dataset_size)
    
    # Dividir usando a mesma metodologia do treinamento
    shuffled_dataset = dataset.shuffle(buffer_size=32)
    train_dataset = shuffled_dataset.take(train_size)
    remaining_dataset = shuffled_dataset.skip(train_size)
    val_dataset = remaining_dataset.take(val_size)
    test_dataset = remaining_dataset.skip(val_size)
    
    # Configurar batch
    test_dataset = test_dataset.batch(batch_size).prefetch(1)
    
    return test_dataset

def calcular_metricas_por_classe(y_true, y_pred, num_classes):
    """
    Calcula IoU, Precision, Recall e F1-score por classe.
    
    Args:
        y_true: Máscaras ground truth (formato one-hot ou sparse)
        y_pred: Máscaras preditas pelo modelo
        num_classes: Número de classes
    
    Returns:
        Dicionário com métricas por classe
    """
    # Converter para formato sparse se estiver em one-hot
    if len(y_true.shape) > 3:
        y_true_sparse = np.argmax(y_true, axis=-1)
    else:
        y_true_sparse = y_true
    
    if len(y_pred.shape) > 3:
        y_pred_sparse = np.argmax(y_pred, axis=-1)
    else:
        y_pred_sparse = y_pred
    
    # Calcular IoU por classe
    mean_iou = MeanIoU(num_classes=num_classes)
    mean_iou.reset_state()
    mean_iou.update_state(y_true_sparse, y_pred_sparse)
    iou_por_classe = mean_iou.result().numpy()
    
    # Reshape para formato linear para calcular precision, recall e f1
    y_true_flat = y_true_sparse.flatten()
    y_pred_flat = y_pred_sparse.flatten()
    
    # Calcular precision, recall e f1 por classe
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true_flat, y_pred_flat, average=None, labels=range(num_classes)
    )
    
    # Calcular métricas globais
    precision_global, recall_global, f1_global, _ = precision_recall_fscore_support(
        y_true_flat, y_pred_flat, average='weighted', labels=range(num_classes)
    )
    
    # Calcular matriz de confusão
    cm = confusion_matrix(y_true_flat, y_pred_flat, labels=range(num_classes))
    
    return {
        'iou_por_classe': iou_por_classe,
        'iou_medio': np.mean(iou_por_classe),
        'precision_por_classe': precision,
        'recall_por_classe': recall,
        'f1_por_classe': f1,
        'precision_global': precision_global,
        'recall_global': recall_global,
        'f1_global': f1_global,
        'matriz_confusao': cm,
        'support': support  # número de amostras por classe
    }

def plotar_matriz_confusao(cm, class_names):
    """
    Plota a matriz de confusão normalizada.
    
    Args:
        cm: Matriz de confusão calculada
        class_names: Lista com nomes das classes
    """
    plt.figure(figsize=(10, 8))
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    
    # Para evitar divisão por zero
    cm_norm = np.nan_to_num(cm_norm)
    
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.ylabel('Classe Real')
    plt.xlabel('Classe Predita')
    plt.title('Matriz de Confusão Normalizada')
    plt.tight_layout()
    plt.savefig('matriz_confusao.png', dpi=300)
    plt.show()

def plotar_metricas_por_classe(metricas, class_names):
    """
    Plota gráficos de barras para IoU, Precision, Recall e F1 por classe.
    
    Args:
        metricas: Dicionário com as métricas calculadas
        class_names: Lista com nomes das classes
    """
    fig, axs = plt.subplots(2, 2, figsize=(15, 10))
    
    # IoU por classe
    axs[0, 0].bar(class_names, metricas['iou_por_classe'])
    axs[0, 0].set_title('IoU por Classe')
    axs[0, 0].set_ylabel('IoU')
    axs[0, 0].set_xticklabels(class_names, rotation=45, ha='right')
    axs[0, 0].axhline(y=0.75, color='r', linestyle='--', label='Min. Aceitável (0.75)')
    axs[0, 0].axhline(y=0.85, color='g', linestyle='--', label='Ideal (0.85)')
    axs[0, 0].legend()
    
    # Precision por classe
    axs[0, 1].bar(class_names, metricas['precision_por_classe'])
    axs[0, 1].set_title('Precision por Classe')
    axs[0, 1].set_ylabel('Precision')
    axs[0, 1].set_xticklabels(class_names, rotation=45, ha='right')
    
    # Recall por classe
    axs[1, 0].bar(class_names, metricas['recall_por_classe'])
    axs[1, 0].set_title('Recall por Classe')
    axs[1, 0].set_ylabel('Recall')
    axs[1, 0].set_xticklabels(class_names, rotation=45, ha='right')
    
    # F1-score por classe
    axs[1, 1].bar(class_names, metricas['f1_por_classe'])
    axs[1, 1].set_title('F1-score por Classe')
    axs[1, 1].set_ylabel('F1-score')
    axs[1, 1].set_xticklabels(class_names, rotation=45, ha='right')
    axs[1, 1].axhline(y=0.80, color='r', linestyle='--', label='Min. Aceitável (0.80)')
    axs[1, 1].axhline(y=0.85, color='g', linestyle='--', label='Ideal (0.85)')
    axs[1, 1].legend()
    
    plt.tight_layout()
    plt.savefig('metricas_por_classe.png', dpi=300)
    plt.show()

def salvar_resultados_tabela(metricas, class_names, arquivo='resultados_metricas.csv'):
    """
    Salva os resultados em um arquivo CSV.
    
    Args:
        metricas: Dicionário com as métricas calculadas
        class_names: Lista com nomes das classes
        arquivo: Nome do arquivo para salvar
    """
    import pandas as pd
    
    # Criar DataFrame com métricas por classe
    df = pd.DataFrame({
        'Classe': class_names,
        'IoU': metricas['iou_por_classe'],
        'Precision': metricas['precision_por_classe'],
        'Recall': metricas['recall_por_classe'],
        'F1-score': metricas['f1_por_classe'],
        'Support': metricas['support']
    })
    
    # Adicionar métricas globais
    df_global = pd.DataFrame({
        'Classe': ['Global'],
        'IoU': [metricas['iou_medio']],
        'Precision': [metricas['precision_global']],
        'Recall': [metricas['recall_global']],
        'F1-score': [metricas['f1_global']],
        'Support': [np.sum(metricas['support'])]
    })
    
    # Concatenar DataFrames
    df_final = pd.concat([df, df_global], ignore_index=True)
    
    # Salvar como CSV
    df_final.to_csv(arquivo, index=False)
    
    # Também salvar como formato LaTeX para o artigo
    latex_table = df_final.to_latex(index=False, float_format="%.4f")
    with open('resultados_metricas.tex', 'w') as f:
        f.write(latex_table)
    
    print(f"Resultados salvos em {arquivo} e resultados_metricas.tex")
    
    return df_final

def avaliar_modelo():
    """Função principal para avaliar o modelo e gerar relatórios."""
    # Parâmetros a serem configurados
    DATA_DIR = "caminho/para/seus/dados/de/teste"  # Substitua pelo caminho correto
    NUM_CLASSES = 10  # Substitua pelo número correto de classes
    CLASS_NAMES = [f"Classe_{i}" for i in range(NUM_CLASSES)]  # Substitua pelos nomes reais
    
    # Carregar o modelo
    modelo = carregar_modelo()
    if modelo is None:
        return
    
    # Preparar dataset de teste
    # dataset_teste = preparar_dataset_teste(DATA_DIR)
    
    # Aqui você precisa implementar a parte de carregamento dos dados de teste e ground truth
    # para sua aplicação específica. Abaixo é apenas um exemplo genérico.
    
    print("Avaliando modelo...")
    
    ### USAR model.evaluate() do TensorFlow ###
    # Isso requer que seu dataset_teste esteja formatado corretamente
    # resultados = modelo.evaluate(dataset_teste, verbose=1)
    # print(f"Resultados da avaliação: {resultados}")
    
    # Para cada batch no dataset de teste
    # for images, masks in dataset_teste:
    #     # Fazer predições
    #     preds = modelo.predict(images)
    #     
    #     # Adicionar aos arrays completos
    #     y_true_all.append(masks.numpy())
    #     y_pred_all.append(preds)
    
    # Concatenar todos os batches
    # y_true_all = np.concatenate(y_true_all, axis=0)
    # y_pred_all = np.concatenate(y_pred_all, axis=0)
    
    # Neste ponto, você deve ter y_true_all e y_pred_all preenchidos com seus dados reais
    # O código abaixo é para mostrar o que seria feito com esses dados
    
    # Substitua este bloco pelo seu código real para carregar os dados de teste
    print("ATENÇÃO: Este é um exemplo. Substitua pelo seu código real para carregar os dados de teste.")
    # Exemplo simulado - remova e use seus dados reais
    # y_true_all = np.random.randint(0, NUM_CLASSES, size=(100, 512, 512))
    # y_pred_all = np.random.random((100, 512, 512, NUM_CLASSES))
    
    # Calcular métricas detalhadas
    # metricas = calcular_metricas_por_classe(y_true_all, y_pred_all, NUM_CLASSES)
    
    # Verificar se atende aos critérios mínimos
    # print(f"\n===== RESULTADOS DA AVALIAÇÃO =====")
    # print(f"IoU Global Médio: {metricas['iou_medio']:.4f} - {'APROVADO' if metricas['iou_medio'] >= 0.75 else 'REPROVADO'}")
    # print(f"F1-score Global: {metricas['f1_global']:.4f} - {'APROVADO' if metricas['f1_global'] >= 0.80 else 'REPROVADO'}")
    
    # Salvar resultados em tabela
    # df_resultados = salvar_resultados_tabela(metricas, CLASS_NAMES)
    # print("\nResultados detalhados por classe:")
    # print(df_resultados)
    
    # Visualizações
    # plotar_matriz_confusao(metricas['matriz_confusao'], CLASS_NAMES)
    # plotar_metricas_por_classe(metricas, CLASS_NAMES)
    
    # print("\nPróximos passos:")
    # print("1. Verifique as visualizações salvas (matriz_confusao.png e metricas_por_classe.png)")
    # print("2. Analise os resultados detalhados em 'resultados_metricas.csv'")
    # print("3. Use o arquivo LaTeX gerado 'resultados_metricas.tex' para seu artigo")

if __name__ == "__main__":
    avaliar_modelo()